# How to find insider buying clusters with the Form 4 API

Pull every open-market insider purchase for a window, group by company and keep the ones where three or more insiders bought. Python, 30 lines.

Read it with charts at https://www.quantisnow.com/sec-api/guides/insider-buying-clusters. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

One insider buying is a data point. Three directors and the CFO buying the same stock in the same two weeks is a cluster, and clusters are the version of the insider signal that holds up in most studies. The Form 4 API has every open-market purchase with its dollar value, so finding them is one query and a group-by.

## Pull every purchase in the window

Transaction code `P` is an open-market or private purchase. Grants (`A`), option exercises (`M`) and tax withholding (`F`) are not buys and would drown the signal, so filter on `P` and on a dollar floor to skip token purchases.

### Which insiders bought more than $1 million of their own company's stock in the last 30 days?

Large open-market buys are the insider signal most studies find worth watching.

```shell
curl -X POST https://www.quantisnow.com/api/v1/insider-trades \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "formType:4 AND transactionCodes:P AND purchased.valueUsd:>1000000 AND filedAt:[2026-10-25 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"purchased.valueUsd": {"order": "desc"}}]
}'
```

Response (real, first 2 of 136):

```json
{
  "total": {
    "value": 136,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001011438-26-000643:0001307631",
      "accessionNo": "0001011438-26-000643",
      "formType": "4",
      "amendment": false,
      "restatesEarlier": false,
      "filedAt": "2026-11-03T16:35:59-05:00",
      "transactionDate": "2026-11-02",
      "issuer": {
        "cik": "0000789570",
        "name": "MGM Resorts International",
        "ticker": "MGM"
      },
      "reportingOwner": {
        "cik": "0001307631",
        "name": "Meister Keith A.",
        "officerTitle": null,
        "isDirector": true,
        "isOfficer": false,
        "isTenPercentOwner": false
      },
      "transactionCodes": [
        "P"
      ],
      "purchased": {
        "shares": 1745592,
        "valueUsd": 51740827.48
      },
      "sold": {
        "shares": 0,
        "valueUsd": 0
      },
      "sharesOwnedAfter": null,
      "rule10b51": false,
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/789570/000101143826000643/0001011438-26-000643-index.htm"
    },
    {
      "id": "0001104659-26-121503:0001820515",
      "accessionNo": "0001104659-26-121503",
      "formType": "4/A",
      "amendment": true,
      "restatesEarlier": false,
      "filedAt": "2026-11-02T17:47:09-05:00",
      "transactionDate": "2026-10-29",
      "issuer": {
        "cik": "0000084839",
        "name": "Rollins Inc",
        "ticker": "ROL"
      },
      "reportingOwner": {
        "cik": "0001820515",
        "name": "R. Randall Rollins Voting Trust U/A dated August 25, 1994",
        "officerTitle": null,
        "isDirector": false,
        "isOfficer": false,
        "isTenPercentOwner": true
      },
      "transactionCodes": [
        "P"
      ],
      "purchased": {
        "shares": 1444402,
        "valueUsd": 49956665.33
      },
      "sold": {
        "shares": 0,
        "valueUsd": 0
      },
      "sharesOwnedAfter": null,
      "rule10b51": false,
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/84839/000110465926121503/0001104659-26-121503-index.htm"
    }
  ]
}
```

## Group by company

In [ ]:
import os, requests
from collections import defaultdict

URL = "https://www.quantisnow.com/api/v1/insider-trades"
KEY = os.environ["QUANTISNOW_API_KEY"]
QUERY = "formType:4 AND transactionCodes:P AND purchased.valueUsd:>25000 AND filedAt:[2026-11-03 TO 2026-11-17]"

rows, start = [], 0
while True:
    page = requests.post(URL, headers={"Authorization": f"Bearer {KEY}"},
                         json={"query": QUERY, "from": start, "size": 50}).json()
    rows += page["data"]
    start += 50
    if start >= page["total"]["value"]:
        break

by_company = defaultdict(lambda: {"people": set(), "usd": 0})
for r in rows:
    c = by_company[r["issuer"]["ticker"]]
    c["people"].add(r["reportingOwner"]["cik"])
    c["usd"] += r["purchased"]["valueUsd"]

clusters = {t: c for t, c in by_company.items() if len(c["people"]) >= 3}
for ticker, c in sorted(clusters.items(), key=lambda kv: -kv[1]["usd"]):
    print(ticker, len(c["people"]), f"${c['usd']:,.0f}")

Count people by `reportingOwner.cik`, not by name. The same director files as "Smith John A" on one form and "SMITH JOHN" on the next; the CIK stays the same.

## Things that trip people up

- Amendments. A 4/A that repeats an earlier purchase comes back as its own record with `restatesEarlier: true`. Add `AND restatesEarlier:false` to count each trade once.
- Ten-percent owners. A fund that owns 10% files Form 4s too. Filter `reportingOwner.isTenPercentOwner:false` if you only want officers and directors.
- Windows over 10,000 rows. A two-week window of purchases never gets close, but if you widen it to a year of all transaction codes, split it by month. See [paging past 10,000 results](https://www.quantisnow.com/sec-api/guides/paginate-past-10000-results).